# C2: Chatbot using an LLM with Prompt Engineering

Design prompts for different tasks (question answering, summarisation, classification), generate responses with an LLM API, evaluate response quality and consistency, and improve performance with prompt engineering.

**Dataset:** a custom FAQ about the NLP lab (30 questions and answers), custom student queries, 20 SST-2 sentences for classification and 3 news articles for summarisation

Cells that call the LLM have no saved output in this file: run them in Colab.

In [1]:
# openai       : one client for both Gemini and OpenRouter (both accept OpenAI-style requests)
# gradio       : web chat window for the chatbot at the end
# scikit-learn : TF-IDF to find the right FAQ entry
# rouge-score  : compare answers with the FAQ answers
!pip install -q openai gradio scikit-learn rouge-score pandas

  Preparing metadata (setup.py) ... done


In [2]:
import time
from collections import Counter
import pandas as pd
from rouge_score import rouge_scorer

## Choose the provider and model

Gemini and OpenRouter both accept requests in the OpenAI format, so the same `openai` client works for both.
Only three things change: the server address (`base_url`), the API key and the model name.

In [3]:
PROVIDER = "gemini"          # "gemini" or "openrouter"

GEMINI_MODEL = "gemini-flash-latest"                         # Google always points this name to its current Flash model
OPENROUTER_MODEL = "meta-llama/llama-3.3-70b-instruct:free"  # any model id from openrouter.ai/models; ":free" ones cost nothing

PAUSE = 4    # seconds to wait after each call, so we stay under the free-tier requests-per-minute limit

In [4]:
from openai import OpenAI
from google.colab import userdata

# provider -> (server address, name of the Colab secret that holds the key, model)
SETTINGS = {
    "gemini":     ("https://generativelanguage.googleapis.com/v1beta/openai/", "GEMINI_API_KEY", GEMINI_MODEL),
    "openrouter": ("https://openrouter.ai/api/v1", "OPENROUTER_API_KEY", OPENROUTER_MODEL),
}
base_url, key_name, MODEL = SETTINGS[PROVIDER]
client = OpenAI(base_url=base_url, api_key=userdata.get(key_name))
print("provider:", PROVIDER, "| model:", MODEL)

provider: gemini | model: gemini-flash-latest


In [5]:
# optional: see which model names this provider offers (if MODEL gives a 404 error, pick one from here)
names = [m.id for m in client.models.list()]
if PROVIDER == "gemini":
    print([n for n in names if "flash" in n])
else:
    print([n for n in names if n.endswith(":free")])

['models/gemini-2.5-flash', 'models/gemini-2.5-flash-preview-tts', 'models/gemini-flash-latest', 'models/gemini-flash-lite-latest', 'models/gemini-2.5-flash-lite', 'models/gemini-2.5-flash-image', 'models/gemini-3-flash-preview', 'models/gemini-3.1-flash-lite-preview', 'models/gemini-3.1-flash-lite', 'models/gemini-3.1-flash-image-preview', 'models/gemini-3.1-flash-image', 'models/gemini-3.1-flash-lite-image', 'models/gemini-3.5-flash', 'models/gemini-3.5-flash-lite', 'models/gemini-omni-flash-preview', 'models/gemini-omni-1.1-flash', 'models/gemini-3.6-flash', 'models/gemini-3.7-flash', 'models/gemini-3.8-flash', 'models/gemini-3.1-flash-tts-preview', 'models/gemini-3.8-flash-tts', 'models/gemini-3.8-flash-lite-tts', 'models/gemini-2.5-flash-native-audio-latest', 'models/gemini-2.5-flash-native-audio-preview-09-2025', 'models/gemini-2.5-flash-native-audio-preview-12-2025', 'models/gemini-3.1-flash-live-preview']


In [6]:
def complete(messages, temperature=0.0):
    # messages: list of {"role": "system" | "user" | "assistant", "content": text}
    for attempt in range(3):
        try:
            response = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature)
            time.sleep(PAUSE)
            return (response.choices[0].message.content or "").strip()
        except Exception as e:                       # usually 429 = too many requests: wait and try again
            print("waiting:", str(e)[:80]); time.sleep(20)
    return ""

def ask(prompt, system=None, temperature=0.0):
    # one question, no history: an optional system message + the user message
    messages = [{"role": "system", "content": system}] if system else []
    messages.append({"role": "user", "content": prompt})
    return complete(messages, temperature)

ask("What is NLP? Answer in one sentence.")

'**Natural Language Processing (NLP)** is a branch of artificial intelligence that enables computers to understand, interpret, manipulate, and generate human language.'

## Datasets: FAQ and custom queries

In [7]:
faq = pd.DataFrame([
    ("How many practicals do I have to perform from each group?", "Any 4 from Group A, any 4 from Group B and any 1 from Group C, plus the mini project."),
    ("Which notebook is for which practical?", "Each notebook is named by its syllabus number: A1 to A5, B1 to B7, C1 and C2. The first cell of every notebook repeats the problem statement from the syllabus."),
    ("What should I write in my journal for a practical?", "The problem statement, the dataset, the main steps with key code, the results tables, and the conclusion. Each notebook ends with a conclusion cell written for the journal and viva questions."),
    ("How do I switch on the GPU in Google Colab?", "Runtime menu, Change runtime type, choose T4 GPU, Save. Do this before running any cell, because changing it restarts the session."),
    ("Why does my notebook forget installed packages and variables?", "A Colab session is a temporary machine. When it disconnects or restarts, run the install cell and all cells above your current cell again."),
    ("Which practicals need a GPU?", "B7 (fine-tuning BERT) and C1 with BART need a GPU to finish in reasonable time. All Group A and the other Group B practicals run on the free CPU runtime."),
    ("How do I get an API key for the LLM practical?", "For Gemini, sign in to Google AI Studio with a Google account and create an API key. For OpenRouter, sign up at openrouter.ai and create a key under Keys. The free tiers are enough for the C2 practical."),
    ("Where should I keep my API key in Colab?", "In Colab Secrets: click the key icon in the left sidebar, add a secret named GEMINI_API_KEY or OPENROUTER_API_KEY, allow notebook access, and read it with userdata.get. Never type the key into a cell that you share."),
    ("I get a 429 error from the LLM API. What does it mean?", "429 means too many requests: the free tier has a rate limit. Wait a minute, send fewer requests, or add a short pause between calls."),
    ("Why does the model give a different answer when I ask the same question again?", "Generation samples from a probability distribution. Lower the temperature (for example 0 to 0.2) for more consistent answers."),
    ("Do I need a Kaggle account to download the datasets?", "No. The notebooks read public copies of the datasets directly from GitHub or the Hugging Face hub."),
    ("Which dataset is used for sentiment analysis practicals?", "The IMDB Movie Reviews dataset of 50,000 reviews, plus the Twitter samples for B6 and SST-2 for B7."),
    ("What is the difference between stemming and lemmatization?", "Stemming cuts word endings with fixed rules and can produce non-words. Lemmatization uses a dictionary and the part of speech to return the real base form, such as be for was."),
    ("What is TF-IDF?", "A weight for each word in a document: how often it appears in the document, multiplied by how rare it is across all documents. Common words like the get low weights."),
    ("What is perplexity?", "A measure of how well a language model predicts held-out text. It is the exponential of the average negative log probability per word; lower is better."),
    ("What is the Viterbi algorithm used for?", "Finding the most probable sequence of hidden states, such as POS tags, in a Hidden Markov Model, using dynamic programming."),
    ("Why is entity-level F1 used for NER instead of token accuracy?", "Most tokens are outside any entity, so token accuracy looks high even for a weak model. Entity-level F1 counts a prediction only if the whole span and the type are correct."),
    ("What is self-attention?", "Each word builds its new vector as a weighted average of all words in the sentence, with weights from the softmax of query-key dot products."),
    ("What is the difference between BERT and GPT?", "BERT is an encoder that reads text in both directions and is used for understanding tasks like classification. GPT-style models are decoders that generate text left to right."),
    ("What does fine-tuning mean?", "Taking a pretrained model, adding a small task layer, and training all weights for a few epochs on labelled task data with a small learning rate."),
    ("What is ROUGE?", "A family of metrics that compare a generated summary with a human reference summary by overlapping words (ROUGE-1), word pairs (ROUGE-2) and the longest common subsequence (ROUGE-L)."),
    ("What is the difference between extractive and abstractive summarization?", "Extractive summarization copies the most important sentences from the text. Abstractive summarization writes new sentences, like a person would."),
    ("What is retrieval-augmented generation?", "Before asking the language model, search your own documents for the passages most relevant to the question and put them in the prompt, so the answer is grounded in those passages."),
    ("What is a hallucination in a language model?", "A fluent answer that states something not supported by the input or not true, such as an invented name, number or reference."),
    ("What is few-shot prompting?", "Putting a few solved examples of the task in the prompt before the real input, so the model copies the format and the kind of answer."),
    ("I get CUDA out of memory while training BERT. What should I do?", "Reduce the batch size or the maximum sequence length, enable fp16, and restart the runtime to free memory from earlier runs."),
    ("spaCy says it cannot find the model en_core_web_sm.", "Run the download cell: python -m spacy download en_core_web_sm, then restart the runtime or load the model again."),
    ("NLTK raises a LookupError for a resource.", "The corpus or model was not downloaded. Run nltk.download with the resource name shown in the error message, for example nltk.download('punkt')."),
    ("What should the mini project include?", "A real-world problem statement, a complete pipeline from raw text to results using classical and transformer-based methods, an evaluation of performance, and a presentation of the results."),
    ("Can I use an LLM API in my mini project?", "Yes, as one part of the pipeline. The syllabus asks you to evaluate system performance, so measure how well the LLM part works, ideally against a simpler method.")],
    columns=["question", "answer"])
faq.head()

,question,answer
0,How many practicals do I have to perform from ...,"Any 4 from Group A, any 4 from Group B and any..."
1,Which notebook is for which practical?,Each notebook is named by its syllabus number:...
2,What should I write in my journal for a practi...,"The problem statement, the dataset, the main s..."
3,How do I switch on the GPU in Google Colab?,"Runtime menu, Change runtime type, choose T4 G..."
4,Why does my notebook forget installed packages...,A Colab session is a temporary machine. When i...


In [8]:
queries = pd.DataFrame([
    ("how many practicals from group B do i need", 0),
    ("how do i turn on the gpu in colab", 3),
    ("where do i get an api key", 6),
    ("i am getting error 429", 8),
    ("why do i get a different answer every time", 9),
    ("what is the difference between stemming and lemmatization", 12),
    ("what is tf-idf", 13),
    ("cuda out of memory while training bert", 25),
    ("when is the end semester exam", None),          # not in the FAQ
    ("what is the capital of australia", None)],     # not in the FAQ
    columns=["query", "faq_row"])
queries

,query,faq_row
0,how many practicals from group B do i need,0.0
1,how do i turn on the gpu in colab,3.0
2,where do i get an api key,6.0
3,i am getting error 429,8.0
4,why do i get a different answer every time,9.0
5,what is the difference between stemming and le...,12.0
6,what is tf-idf,13.0
7,cuda out of memory while training bert,25.0
8,when is the end semester exam,NaN
9,what is the capital of australia,NaN


## (a, b) Simple prompts for the three tasks

In [9]:
# 1. question answering
for q in queries["query"][:3]:
    print("Q:", q); print("A:", ask(q)); print()

Q: how many practicals from group B do i need
A: To give you the exact number, I need a little more context! 

Could you please share:

1. **The subject** (e.g., Physics, Chemistry, Biology, Computer Science, etc.)
2. **The exam board or institution** (e.g., AQA, Edexcel, OCR, Cambridge, or your specific university/college)
3. **The qualification level** (e.g., GCSE, A-Level, BTEC, Undergraduate)

Once you let me know, I can check the exact requirements for your course.

Q: how do i turn on the gpu in colab
A: To turn on the GPU in Google Colab, follow these steps:

1. Open your Google Colab notebook.
2. In the top menu, click on **Runtime**.
3. Select **Change runtime type**.
4. Under **Hardware accelerator**, select **GPU** (usually **T4 GPU** on the free tier).
5. Click **Save**.

---

### How to verify it's working:
You can check if the GPU is connected by running this command in a new code cell:

```python
!nvidia-smi
```

If it is working, it will output a table showing the GPU m

In [10]:
# 2. summarisation
news = pd.read_csv("https://raw.githubusercontent.com/sunnysai12345/News_Summary/master/news_summary.csv", encoding="latin-1").dropna()
news = news[news["ctext"].str.split().str.len().between(200, 500)].sample(3, random_state=1)
articles, references = list(news["ctext"]), list(news["text"])
print(articles[0][:300], "...")

India's longest river bridge, capable of withstanding the weight of a 60-tonne battle tank, will be inaugurated in Assam close to the border with China on May 26 by Prime Minister Narendra Modi.With the inauguration of the 9.15-km-long Dhola-Sadiya bridge over the Brahmaputra river, the prime minist ...


In [11]:
summaries = [ask("Summarize this article:\n" + a) for a in articles]
print(summaries[0])

**Summary:**

On May 26, Prime Minister Narendra Modi will inaugurate the Dhola-Sadiya bridge in Assam, marking both three years of the NDA government and the opening of India’s longest bridge. 

**Key Details:**
* **Scale and Cost:** Spanning 9.15 km across the Brahmaputra River—surpassing Mumbai's Bandra-Worli Sea Link—the bridge was built at a cost of ₹950 crore.
* **Strategic Military Importance:** Located less than 100 km from the Chinese border, the bridge is designed to support 60-tonne battle tanks, enabling rapid deployment of troops and artillery to the Sino-Indian border during conflicts.
* **Civilian Impact:** It connects Assam and Arunachal Pradesh, reducing travel time between the two states by up to four hours and providing Arunachal residents easier access to the nearest airport (Dibrugarh) and railway station (Tinsukia).


In [12]:
# 3. classification
sst = pd.read_csv("https://raw.githubusercontent.com/clairett/pytorch-sentiment-classification/master/data/SST2/test.tsv",
                  sep="\t", header=None, names=["text", "label"], quoting=3).sample(20, random_state=0)
sst["gold"] = sst["label"].map({1: "positive", 0: "negative"})
sst.head(3)

,text,label,gold
792,smith 's point is simple and obvious people 's...,1,positive
655,"for its seriousness , high literary aspiration...",1,positive
118,imagine if you will a tony hawk skating video ...,0,negative


In [13]:
labels = [ask("Is this movie review positive or negative?\n" + t) for t in sst["text"]]
labels[:5]

['This review is **positive** (leaning positive). \n\nWhile the reviewer notes a minor criticism—that the director\'s central point is "simple and obvious"—the word "but" turns the critique around to end on a favorable note, concluding that the film\'s subjects are winning and delightful ("charmers").',
 'This movie review is **positive**. \n\nKey phrases like "high literary aspirations," "stunning acting," and "can only be applauded" clearly indicate strong praise for the film.',
 'This review is almost certainly **negative** (or at the very least, derisive and mocking). \n\nThe reviewer uses chaotic and tacky comparisons—disorienting war-movie action, a mid-2000s skateboarding video, and obnoxious "Jersey Shore techno"—to paint the film as loud, hyperactive, aesthetically cheap, and overwhelming.',
 'This movie review is **positive**. \n\nDescribing a movie as a "feel-good picture in the best sense of the term" means the reviewer found it genuinely uplifting, heartwarming, and well-m

## (c) Evaluate quality and consistency

- **QA:** ROUGE-L between the answer and the FAQ answer; for questions not in the FAQ the right answer is "I don't know".
- **Classification:** is the reply exactly "positive" or "negative" (format), and is it correct (accuracy)?
- **Summarisation:** ROUGE against the human summary, and length.
- **Consistency:** ask the same thing 5 times and count how often the most common reply comes back.

In [14]:
rl = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

def evaluate(qa_answers, cls_labels, sums):
    in_faq = queries["faq_row"].notna()
    qa_score = [rl.score(faq["answer"][int(r)], a)["rougeL"].fmeasure for a, r in zip(qa_answers, queries["faq_row"]) if pd.notna(r)]
    says_dont_know = [("don't know" in a.lower() or "do not know" in a.lower()) for a, ok in zip(qa_answers, in_faq) if not ok]
    clean = [l.strip().strip(".").lower() for l in cls_labels]
    return {"QA ROUGE-L": sum(qa_score) / len(qa_score),
            "QA says I don't know when it should": sum(says_dont_know) / len(says_dont_know),
            "CLS correct format": sum(c in ("positive", "negative") for c in clean) / len(clean),
            "CLS accuracy": sum(c == g for c, g in zip(clean, sst["gold"])) / len(clean),
            "SUM ROUGE-1": sum(rl.score(r, s)["rouge1"].fmeasure for r, s in zip(references, sums)) / len(sums),
            "SUM words": sum(len(s.split()) for s in sums) / len(sums)}

qa_answers = [ask(q) for q in queries["query"]]
results = {"simple prompts": evaluate(qa_answers, labels, summaries)}
pd.DataFrame(results).round(3)

,simple prompts
QA ROUGE-L,0.093
QA says I don't know when it should,0.000
CLS correct format,0.000
CLS accuracy,0.000
SUM ROUGE-1,0.472
SUM words,113.333


In [15]:
def consistency(prompt, temperature, n=5):
    replies = [ask(prompt, temperature=temperature) for _ in range(n)]
    return Counter(r.lower() for r in replies).most_common(1)[0][1] / n, replies

for temp in [1.0, 0.0]:
    score, replies = consistency("Is this movie review positive or negative?\n" + sst["text"].iloc[0], temp)
    print(f"temperature {temp}: consistency {score:.0%}", replies)

temperature 1.0: consistency 20% ['This review is **positive**. \n\nAlthough the reviewer notes that the film\'s premise is "simple and obvious," the pivot word "but" shifts the final verdict to praise, concluding that the film works because the subjects are "charmers."', 'This review is generally **positive** (or mildly positive). \n\nWhile the critic starts with a mild critique—noting that the filmmaker\'s point is "simple and obvious"—the pivot word **"but"** shifts the focus to the redeeming quality of the film: the subjects are **"charmers."** In film criticism, when the charm of the subjects outweighs the simplicity of the premise, it lands as a positive recommendation.', 'This review is **positive** (or leans positive). \n\nWhile the reviewer notes that the premise is "simple and obvious," the word **"but"** shifts the focus to the redeeming, enjoyable quality of the film: the subjects are **"charmers."** In film criticism, praising the subjects as charming despite a simple prem

## (d) Improve with prompt engineering

- **Role** (system instruction), **clear task**, **output format**, **few-shot examples** for classification,
- **length limit** and "keep the facts" for summarisation,
- **grounding (RAG)** for QA: find the 3 most similar FAQ entries with TF-IDF and tell the model to answer only from them.

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer(stop_words="english")
faq_vecs = vec.fit_transform(faq["question"] + " " + faq["answer"])

def retrieve(question, k=3):
    sims = (faq_vecs @ vec.transform([question]).T).toarray().ravel()
    return faq.iloc[sims.argsort()[::-1][:k]]

retrieve("how do i turn on the gpu in colab")

,question,answer
3,How do I switch on the GPU in Google Colab?,"Runtime menu, Change runtime type, choose T4 G..."
5,Which practicals need a GPU?,B7 (fine-tuning BERT) and C1 with BART need a ...
7,Where should I keep my API key in Colab?,In Colab Secrets: click the key icon in the le...


In [17]:
SYSTEM = "You are the teaching assistant for the NLP lab. Answer briefly and only from the FAQ given."

def qa_prompt(q):
    context = "\n".join(f"Q: {r.question}\nA: {r.answer}" for r in retrieve(q).itertuples())
    return (f"FAQ:\n{context}\n\nAnswer the question using only the FAQ above. "
            f"If the FAQ does not contain the answer, reply: I don't know, please ask your teacher.\n\nQuestion: {q}")

examples = "Review: a gripping story with brilliant acting\nLabel: positive\n\nReview: two hours i will never get back\nLabel: negative"
def cls_prompt(t):
    return f"Classify the movie review as positive or negative. Reply with one word only.\n\n{examples}\n\nReview: {t}\nLabel:"

def sum_prompt(a):
    return f"Summarise the news article in at most 60 words. Keep names and numbers exactly as in the article; add nothing.\n\nArticle:\n{a}"

print(qa_prompt("how do i turn on the gpu in colab"))

FAQ:
Q: How do I switch on the GPU in Google Colab?
A: Runtime menu, Change runtime type, choose T4 GPU, Save. Do this before running any cell, because changing it restarts the session.
Q: Which practicals need a GPU?
A: B7 (fine-tuning BERT) and C1 with BART need a GPU to finish in reasonable time. All Group A and the other Group B practicals run on the free CPU runtime.
Q: Where should I keep my API key in Colab?
A: In Colab Secrets: click the key icon in the left sidebar, add a secret named GEMINI_API_KEY or OPENROUTER_API_KEY, allow notebook access, and read it with userdata.get. Never type the key into a cell that you share.

Answer the question using only the FAQ above. If the FAQ does not contain the answer, reply: I don't know, please ask your teacher.

Question: how do i turn on the gpu in colab


In [18]:
qa2 = [ask(qa_prompt(q), system=SYSTEM) for q in queries["query"]]
labels2 = [ask(cls_prompt(t)) for t in sst["text"]]
sums2 = [ask(sum_prompt(a)) for a in articles]

results["engineered prompts"] = evaluate(qa2, labels2, sums2)
pd.DataFrame(results).round(3)

,simple prompts,engineered prompts
QA ROUGE-L,0.093,0.907
QA says I don't know when it should,0.000,1.000
CLS correct format,0.000,1.000
CLS accuracy,0.000,0.950
SUM ROUGE-1,0.472,0.527
SUM words,113.333,53.000


In [19]:
pd.DataFrame({"query": queries["query"], "simple prompt": qa_answers, "with FAQ (RAG)": qa2})

,query,simple prompt,with FAQ (RAG)
0,how many practicals from group B do i need,"To tell you the exact number, I need a bit mor...",Any 4 practicals from Group B.
1,how do i turn on the gpu in colab,"To turn on the GPU in Google Colab, follow the...","Runtime menu, Change runtime type, choose T4 G..."
2,where do i get an api key,"To get an API key, you need to generate one di...","For Gemini, sign in to Google AI Studio with a..."
3,i am getting error 429,**Error 429: Too Many Requests** means that yo...,429 means too many requests: the free tier has...
4,why do i get a different answer every time,You get a different answer every time because ...,Generation samples from a probability distribu...
5,what is the difference between stemming and le...,Both **stemming** and **lemmatization** are te...,Stemming cuts word endings with fixed rules an...
6,what is tf-idf,**TF-IDF** stands for **Term Frequency-Inverse...,A weight for each word in a document: how ofte...
7,cuda out of memory while training bert,"The **""CUDA out of memory"" (OOM)** error is ex...",Reduce the batch size or the maximum sequence ...
8,when is the end semester exam,I don't have access to your specific schedule!...,"I don't know, please ask your teacher."
9,what is the capital of australia,The capital of Australia is **Canberra**.,"I don't know, please ask your teacher."


## The chatbot

The API does not remember anything between calls. A chatbot "remembers" because we send the whole
conversation every time: system message + all earlier turns + the new question.

In [20]:
def chat_reply(question, history):
    # history: earlier turns as [{"role": "user", ...}, {"role": "assistant", ...}, ...]
    messages = [{"role": "system", "content": SYSTEM}] + history
    messages.append({"role": "user", "content": qa_prompt(question)})   # new question + the matching FAQ entries
    return complete(messages, temperature=0.2)

history = []
for q in ["How do I switch on the GPU in Colab?", "Which practicals need it?", "When is the exam?"]:
    answer = chat_reply(q, history)
    history += [{"role": "user", "content": q}, {"role": "assistant", "content": answer}]   # store the plain question, not the long prompt
    print("You:", q); print("Bot:", answer); print()

print(len(history), "messages in history")

You: How do I switch on the GPU in Colab?
Bot: Runtime menu, Change runtime type, choose T4 GPU, Save. Do this before running any cell, because changing it restarts the session.

You: Which practicals need it?
Bot: B7 (fine-tuning BERT) and C1 with BART need a GPU to finish in reasonable time. All Group A and the other Group B practicals run on the free CPU runtime.

You: When is the exam?
Bot: I don't know, please ask your teacher.

6 messages in history


In [21]:
# talk to it yourself; type quit to stop
history = []
while True:
    q = input("You: ")
    if q.lower() == "quit":
        break
    answer = chat_reply(q, history)
    history += [{"role": "user", "content": q}, {"role": "assistant", "content": answer}]
    print("Bot:", answer)

You: hi
Bot: I don't know, please ask your teacher.
You: how many practicals needed for group A
Bot: Any 4 from Group A.
You: quit


## The chatbot as a web app (Gradio)

`gr.ChatInterface` draws the chat window and keeps the history for us. It calls our function with the new
message and the history so far, and shows whatever text the function returns.

In [22]:
import gradio as gr

def text_of(content):
    # newer Gradio sends a message as a list of parts like [{"type": "text", "text": "..."}]; older versions send plain text
    if isinstance(content, str):
        return content
    return " ".join(part.get("text", "") for part in content if isinstance(part, dict))

def respond(message, history):
    # Gradio's history items also carry extra keys (metadata, options); keep only role and plain text for the API
    history = [{"role": m["role"], "content": text_of(m["content"])} for m in history]
    return chat_reply(message, history)

# quick check without the web page: one earlier turn in the newer Gradio format
respond("Which practicals need it?",
        [{"role": "user", "content": [{"type": "text", "text": "How do I switch on the GPU in Colab?"}]},
         {"role": "assistant", "content": [{"type": "text", "text": "Runtime, Change runtime type, T4 GPU."}]}])

'B7 (fine-tuning BERT) and C1 with BART need a GPU to finish in reasonable time.'

In [23]:
demo = gr.ChatInterface(
    fn=respond,
    title="NLP Lab Assistant",
    description="Ask about the NLP practicals. Answers come only from the lab FAQ.",
    examples=["How do I switch on the GPU in Colab?", "What is TF-IDF?", "I am getting error 429"],
)
demo.launch(share=True, debug=True)    # share=True also prints a public link (opens on a phone); stop the cell to close the app

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1d4ecde8f82f7f854f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://1d4ecde8f82f7f854f.gradio.live


## Conclusion
- Prompts for QA, summarisation and classification were sent to an LLM (Gemini or a model on OpenRouter) through the OpenAI Python SDK; only the server address, key and model name change between providers.
- What to look for in your table: simple prompts tend to give long, chatty answers, guess answers to questions about our lab, and return sentences instead of labels.
- Each prompt-engineering change targets one of these: a role and a one-word format make labels usable, few-shot examples show the decision, a length limit controls summaries, and grounding in retrieved FAQ entries makes the bot answer from our data and say "I don't know" otherwise. The before/after table shows how much each task improved.
- Temperature 0 makes answers repeatable; at 1.0 the same prompt can give different answers.
- The model itself is stateless: the chatbot keeps the conversation by sending the full history with every call, and Gradio only adds a web page on top of the same function.